# 01. Data Loading
## 📚 Learning Objectives

By completing this notebook, you will:
- Load data from CSV files with advanced options
- Load from Excel files and multiple sheets
- Load from JSON (including nested structures)
- Handle large files using chunking
- Implement error handling for robust data loading

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

---


**All concepts are explained in the code comments below - you can learn everything from this notebook alone!**

---

## 📚 Prerequisites (What You Need First)

**BEFORE starting this notebook**, you should have completed:
- ✅ **Unit 1: All examples** - You need pandas, NumPy, cuDF basics
- ✅ **Understanding of DataFrames and basic file operations**
- ✅ **Knowledge of different file formats** (CSV, Excel, JSON)

**If you haven't completed these**, you might struggle with:
- Understanding DataFrame operations
- Knowing which loading method to use
- Handling file paths and errors

---

## 🔗 Where This Notebook Fits

**This is the FIRST example in Unit 2** - Data Cleaning and Preparation!

**Why this example FIRST in Unit 2?**
- **Before** you can clean data, you need to load it from files
- **Before** you can analyze data, you need to load it correctly
- **Before** you can handle large datasets, you need chunking strategies

**Builds on**: 
- 📓 Unit 1: pandas DataFrame knowledge

**Leads to**: 
- 📓 Example 2: Missing Values & Duplicates (needs data loading skills)
- 📓 Example 3: Outliers & Transformation (needs loaded data)
- 📓 All cleaning operations (all need data loaded first!)

**Why this order?**
1. Data loading is the first step (can't clean what you don't have)
2. Advanced loading teaches chunking (essential for large data)
3. Error handling in loading (foundation for robust code)

---

## The Story: Getting Your Ingredients

Imagine you're cooking. **Before** you can clean and prepare ingredients, you need to get 
them from the store - know what's available, handle different packages, check if items 
are missing. **After** getting everything loaded, you can start preparing!

Same with data science: **Before** cleaning and analyzing, we load data from files - know 
file formats, handle different sources, check for errors. **After** loading successfully, 
we can start cleaning!

---

## Why Advanced Data Loading Matters

Advanced loading techniques are essential because:
- **Multiple Formats**: Real data comes in CSV, Excel, JSON, databases
- **Large Files**: Need chunking to load huge datasets that don't fit in memory
- **Error Handling**: Files may be missing, corrupted, or have encoding issues
- **Performance**: Correct loading options make your code faster

## Learning Objectives
1. Load data from CSV files with advanced options
2. Load data from Excel files and multiple sheets
3. Load data from JSON files (including nested structures)
4. Handle large files using chunking techniques
5. Implement error handling for robust data loading

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries: pandas
- File paths: CSV, Excel, JSON (notebook creates sample files in `unit2-cleaning/examples/`)

**Outputs:** What you'll see when you run the cells

- Loaded DataFrames (sample_data, large_data chunked)
- Printed success messages and basic stats

---

In [1]:
# WHAT: Import the file-handling toolkit - pandas for tables, json and os/pathlib for files.
# WHY: Loading is the first step of every project; these libraries cover the three most common formats (CSV, Excel, JSON).

# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

print("✅ Libraries imported successfully!")
print("\n📚 What each library does:")
print("   - pandas: Load data from CSV, Excel, JSON files")
print("   - numpy: Generate sample data for examples")
print("   - json: Handle JSON file formats")
print("   - os/pathlib: Handle file paths and directories")

print("\n" + "=" * 70)
print("=" * 70)
print("\n📚 Prerequisites: Unit 1 completed, pandas DataFrame knowledge")
print("🔗 This is the FIRST example in Unit 2 - foundation for data cleaning")
print("🎯 Goal: Master loading data from multiple sources and formats\n")

✅ Libraries imported successfully!

📚 What each library does:
   - pandas: Load data from CSV, Excel, JSON files
   - numpy: Generate sample data for examples
   - json: Handle JSON file formats
   - os/pathlib: Handle file paths and directories


📚 Prerequisites: Unit 1 completed, pandas DataFrame knowledge
🔗 This is the FIRST example in Unit 2 - foundation for data cleaning
🎯 Goal: Master loading data from multiple sources and formats



## Part 1: Loading from CSV Files

**BEFORE**: You know basic CSV loading but not advanced options.

**AFTER**: You'll load CSV files with encoding, missing value handling, and date parsing!

**Why this matters**: Real CSV files have encoding issues, missing values, and date formats!

## Step 1: Loading from CSV

**BEFORE**: We need sample data, but we don't have any CSV files yet.

**AFTER**: We'll create sample CSV data and load it with different options!

In [2]:
# WHAT: Create a 100-row sample CSV on disk, then load it back with default and with explicit options.
# WHY: Real CSV loading needs encoding, NA markers, and date parsing - defaults silently give you strings where you wanted dates.

print("\n1. Loading from CSV")
print("-" * 70)
# Create sample CSV data
sample_data = {
'id': range(1, 101),
'name': [f'Person_{i}' for i in range(1, 101)],
'age': np.random.randint(18, 80, 100),
'salary': np.random.normal(50000, 15000, 100),
'department': np.random.choice(['IT', 'HR', 'Finance', 'Sales'], 100),
'join_date': pd.date_range('2020-01-01', periods=100, freq='D')
}
df_sample = pd.DataFrame(sample_data)
csv_file = 'unit2-cleaning/examples/sample_data.csv'
os.makedirs(os.path.dirname(csv_file), exist_ok=True)
df_sample.to_csv(csv_file, index=False)
print(f"✓ Created sample CSV file: {csv_file}")
# Load CSV with different options
print("\nLoading CSV with default options")
df1 = pd.read_csv(csv_file)
print(f"Shape: {df1.shape}")
print("\nLoading CSV with specific options")
df2 = pd.read_csv(csv_file, encoding='utf-8',
na_values=['', 'NULL', 'null'],
parse_dates=['join_date'])
print(f"Shape: {df2.shape}")


1. Loading from CSV
----------------------------------------------------------------------
✓ Created sample CSV file: unit2-cleaning/examples/sample_data.csv

Loading CSV with default options
Shape: (100, 6)

Loading CSV with specific options
Shape: (100, 6)


## Part 2: Loading from Excel Files

**BEFORE**: You can load CSV but not Excel files.

**AFTER**: You'll load Excel files and handle multiple sheets!

**Why this matters**: Many organizations use Excel - you need to handle .xlsx files!

## Step 2: Loading from Excel

**BEFORE**: We have CSV data but need Excel format.

**AFTER**: We'll create Excel files and load from specific sheets!

In [3]:
# WHAT: Write a two-sheet Excel workbook, list its sheets, and load them one-by-one and all at once.
# WHY: Spreadsheets are how business data usually arrives - sheet_name (str, list, or None) covers every loading case.

print("\n\n2. Loading from Excel")
print("-" * 70)
excel_file = 'unit2-cleaning/examples/sample_data.xlsx'

# Create a workbook with TWO sheets: employees + departments
df_departments = pd.DataFrame({
    'dept_id': [1, 2, 3],
    'dept_name': ['Engineering', 'Sales', 'HR'],
    'budget_k': [900, 500, 250],
})
with pd.ExcelWriter(excel_file) as writer:
    df_sample.to_excel(writer, index=False, sheet_name='Employees')
    df_departments.to_excel(writer, index=False, sheet_name='Departments')
print(f"✓ Created Excel file with 2 sheets: {excel_file}")

# Discover which sheets a workbook contains
sheets = pd.ExcelFile(excel_file).sheet_names
print(f"\nSheets in the workbook: {sheets}")

# Load a sheet by name
print("\nLoading sheet by name ('Employees'):")
df_excel = pd.read_excel(excel_file, sheet_name='Employees')
print(f"Shape: {df_excel.shape}")
print(f"Columns: {list(df_excel.columns)}")

# Load a different sheet
print("\nLoading the second sheet ('Departments'):")
df_dept = pd.read_excel(excel_file, sheet_name='Departments')
print(df_dept.to_string(index=False))

# Load ALL sheets at once: sheet_name=None returns a dict of DataFrames
print("\nLoading ALL sheets at once (sheet_name=None):")
all_sheets = pd.read_excel(excel_file, sheet_name=None)
for name, sheet_df in all_sheets.items():
    print(f"  '{name}': {sheet_df.shape[0]} rows x {sheet_df.shape[1]} columns")



2. Loading from Excel
----------------------------------------------------------------------
✓ Created Excel file with 2 sheets: unit2-cleaning/examples/sample_data.xlsx

Sheets in the workbook: ['Employees', 'Departments']

Loading sheet by name ('Employees'):
Shape: (100, 6)
Columns: ['id', 'name', 'age', 'salary', 'department', 'join_date']

Loading the second sheet ('Departments'):
 dept_id   dept_name  budget_k
       1 Engineering       900
       2       Sales       500
       3          HR       250

Loading ALL sheets at once (sheet_name=None):
  'Employees': 100 rows x 6 columns
  'Departments': 3 rows x 3 columns


## Part 3: Loading from JSON Files

**BEFORE**: You can load CSV/Excel but not JSON (API data).

**AFTER**: You'll load JSON files and handle nested structures!

**Why this matters**: APIs return JSON - web data is often in JSON format!

## Step 3: Loading from JSON

**BEFORE**: We have structured data but need JSON format (like APIs).

**AFTER**: We'll create JSON files and load nested data structures!

In [4]:
# WHAT: Write a nested JSON file, then load it with read_json and flatten it with json_normalize.
# WHY: APIs return nested JSON; json_normalize is the standard way to turn nested records into a flat table.

print("\n\n3. Loading from JSON")
print("-" * 70)
# Create sample JSON data
json_data = {
'employees': [
{'id': i, 'name': f'Employee_{i}', 'age': np.random.randint(25, 55)}
for i in range(1, 21)
]
}
json_file = 'unit2-cleaning/examples/sample_data.json'
with open(json_file, 'w') as f:
    json.dump(json_data, f, indent=2)
print(f"✓ Created sample JSON file: {json_file}")
# Load JSON
print("\nLoading JSON file")
df_json = pd.read_json(json_file)
print(f"Shape: {df_json.shape}")
# Load JSON with nested structure
df_json2 = pd.json_normalize(json_data, record_path='employees')
print(f"\nNormalized JSON shape: {df_json2.shape}")



3. Loading from JSON
----------------------------------------------------------------------
✓ Created sample JSON file: unit2-cleaning/examples/sample_data.json

Loading JSON file
Shape: (20, 1)

Normalized JSON shape: (20, 3)


## Part 4: Handling Large Files - Chunking

**BEFORE**: You can load small files but not large ones (memory issues).

**AFTER**: You'll load huge files in chunks without running out of memory!

**Why this matters**: Large datasets don't fit in memory - chunking is essential!

## Step 4: Chunking Large Files

**BEFORE**: Large files crash our program (out of memory).

**AFTER**: We'll process large files in manageable chunks!

In [5]:
# WHAT: Write a 100k-row CSV and load it back in 10k-row chunks with chunksize.
# WHY: Chunking keeps memory bounded - it is how you process files bigger than RAM.

print("\n\n4. Handling Large Files (Chunking)")
print("-" * 70)
# Create larger dataset
large_data = {
'id': range(1, 100001),
'value': np.random.randn(100000), 'category': np.random.choice(['A', 'B', 'C'], 100000)
}
df_large = pd.DataFrame(large_data)
large_csv = 'unit2-cleaning/examples/large_data.csv'
df_large.to_csv(large_csv, index=False)
print(f"✓ Created large CSV file: {large_csv} ({len(df_large):,} rows)")
# Load in chunks
print("\nLoading in chunks")
chunk_size = 10000
chunks = []
for chunk in pd.read_csv(large_csv, chunksize=chunk_size):
    chunks.append(chunk)
    print(f"  Loaded chunk with {len(chunk)} rows")
df_chunked = pd.concat(chunks, ignore_index=True)
print(f"\n✓ Total rows loaded: {len(df_chunked):,}")



4. Handling Large Files (Chunking)
----------------------------------------------------------------------
✓ Created large CSV file: unit2-cleaning/examples/large_data.csv (100,000 rows)

Loading in chunks
  Loaded chunk with 10000 rows
  Loaded chunk with 10000 rows
  Loaded chunk with 10000 rows
  Loaded chunk with 10000 rows
  Loaded chunk with 10000 rows
  Loaded chunk with 10000 rows
  Loaded chunk with 10000 rows
  Loaded chunk with 10000 rows
  Loaded chunk with 10000 rows
  Loaded chunk with 10000 rows

✓ Total rows loaded: 100,000


## Part 5: Error Handling

**BEFORE**: Your code crashes when files are missing or corrupted.

**AFTER**: You'll handle errors gracefully and keep your code running!

**Why this matters**: Real-world files are missing, corrupted, or have wrong formats!

## Step 5: Robust Error Handling

**BEFORE**: Missing files cause crashes.

**AFTER**: We'll create functions that handle errors gracefully!

In [6]:
# WHAT: Define safe_load_csv - a loader wrapped in try/except for the common failure modes.
# WHY: Production code must survive missing or empty files; catching specific exceptions gives useful messages instead of crashes.

def safe_load_csv(filepath, **kwargs):
    """Safely load CSV file with error handling."""
    try:
        df = pd.read_csv(filepath, **kwargs)
        print(f"✓ Successfully loaded {len(df)} rows from {filepath}")
        return df
    except FileNotFoundError:
        print(f"✗ Error: File {filepath} not found")
        return None
    except pd.errors.EmptyDataError:
        print(f"✗ Error: File {filepath} is empty")
        return None
    except Exception as e:
        print(f"✗ Error loading {filepath}: {str(e)}")
        return None

print("✓ Helper safe_load_csv defined")

✓ Helper safe_load_csv defined


In [7]:
# WHAT: Exercise the safe loader on a missing file and on a real one.
# WHY: Testing BOTH the failure and the success path proves the error handling actually works.

# Safely load CSV with error handling
print("\nTesting error handling")
safe_load_csv('nonexistent_file.csv')
safe_load_csv(csv_file)  # Should succeed


Testing error handling
✗ Error: File nonexistent_file.csv not found
✓ Successfully loaded 100 rows from unit2-cleaning/examples/sample_data.csv


,id,name,age,salary,department,join_date
0,1,Person_1,48,72482.932596,Sales,2020-01-01
1,2,Person_2,51,84641.070826,Finance,2020-01-02
2,3,Person_3,36,43366.069042,Finance,2020-01-03
3,4,Person_4,45,29265.878180,IT,2020-01-04
4,5,Person_5,50,33476.773837,Finance,2020-01-05
...,...,...,...,...,...,...
95,96,Person_96,37,52630.421884,IT,2020-04-05
96,97,Person_97,38,40310.484449,HR,2020-04-06
97,98,Person_98,77,70033.862946,HR,2020-04-07
98,99,Person_99,65,30527.147137,HR,2020-04-08


## 🎯 Summary: What We Learned

In [8]:
# WHAT: Print a recap of the loading techniques covered.
# WHY: A summary in one place helps you pick the right loading tool per format and file size.

print("\n" + "=" * 70)
print("🎯 SUMMARY: What We Learned")
print("=" * 70)

print("\n📋 BEFORE this notebook:")
print("   - You could load basic CSV files but not advanced formats")
print("   - You didn't know how to handle large files (memory issues)")
print("   - Your code crashed when files were missing or corrupted")

print("\n✅ AFTER this notebook:")
print("   - You can load CSV, Excel, and JSON files with advanced options")
print("   - You can handle large files using chunking (no memory issues)")
print("   - You can handle errors gracefully (robust code)")
print("   - You know when to use each loading method")

print("\n📚 Key Concepts Covered:")
print("   1. CSV Loading (encoding, missing values, date parsing)")
print("   2. Excel Loading (multiple sheets, specific sheet selection)")
print("   3. JSON Loading (nested structures, normalization)")
print("   4. Chunking Large Files (process in parts, avoid memory issues)")
print("   5. Error Handling (robust loading, handle missing/corrupted files)")

print("\n🔗 Where Data Loading Fits:")
print("   - FIRST step in any data science project (need data before analysis)")
print("   - Foundation for all cleaning operations (must load before cleaning)")
print("   - Essential for production pipelines (robust error handling)")

print("\n➡️  Next Steps:")
print("   - Continue to Example 2: Missing Values & Duplicates")
print("   - You'll learn how to clean data AFTER loading it")
print("   - Loading skills are essential for the cleaning techniques!")

print("\n✓ Sample files created in 'unit2-cleaning/examples' directory")
print("\n" + "=" * 70)


🎯 SUMMARY: What We Learned

📋 BEFORE this notebook:
   - You could load basic CSV files but not advanced formats
   - You didn't know how to handle large files (memory issues)
   - Your code crashed when files were missing or corrupted

✅ AFTER this notebook:
   - You can load CSV, Excel, and JSON files with advanced options
   - You can handle large files using chunking (no memory issues)
   - You can handle errors gracefully (robust code)
   - You know when to use each loading method

📚 Key Concepts Covered:
   1. CSV Loading (encoding, missing values, date parsing)
   2. Excel Loading (multiple sheets, specific sheet selection)
   3. JSON Loading (nested structures, normalization)
   4. Chunking Large Files (process in parts, avoid memory issues)
   5. Error Handling (robust loading, handle missing/corrupted files)

🔗 Where Data Loading Fits:
   - FIRST step in any data science project (need data before analysis)
   - Foundation for all cleaning operations (must load before cleanin

## 🚫 When Data Loading Hits a Dead End

**BEFORE**: We've successfully loaded data from multiple sources.

**AFTER**: We discover the loaded data has problems - missing values, duplicates, and quality issues!

**Why this matters**: Loading data is just the first step - real-world data is messy and needs cleaning!

---

### The Problem We've Discovered

We've learned:
- ✅ How to load data from CSV, Excel, and JSON
- ✅ How to handle large files with chunking
- ✅ How to handle errors during loading

**But we have a problem:**
- ❓ **What if the loaded data has missing values?**
- ❓ **What if there are duplicate rows?**
- ❓ **What if the data quality is poor?**

**The Dead End:**
- We can load data successfully
- But the data itself has quality issues
- We can't proceed with analysis until we clean the data!

---

### Demonstrating the Problem

Let's check the data we just loaded - does it have quality issues?

In [9]:
# WHAT: Inspect the freshly loaded data for missing values and duplicates (injecting some if none exist).
# WHY: Loading is not the finish line - this preview of quality problems motivates the cleaning techniques of the next notebook.

print("\n" + "=" * 70)
print("🚫 DEMONSTRATING THE DEAD END: Data Quality Issues")
print("=" * 70)

# Load the data we created earlier
df_check = pd.read_csv(csv_file)

# Check for missing values
missing_count = df_check.isnull().sum().sum()
missing_per_col = df_check.isnull().sum()

print(f"\n📊 Checking data quality in loaded dataset...")
print(f"   Dataset shape: {df_check.shape[0]} rows × {df_check.shape[1]} columns")

print(f"\n⚠️  Missing Values Found:")
if missing_count > 0:
    print(f"   - Total missing values: {missing_count}")
    print(f"   - Missing values per column:")
    for col, count in missing_per_col.items():
        if count > 0:
            print(f"     • {col}: {count} missing ({count/len(df_check)*100:.1f}%)")
else:
    # Create a version with missing values to demonstrate the problem
    print(f"   - No missing values in current dataset")
    print(f"   - But real-world data often has missing values!")
    # Add some missing values to demonstrate
    df_with_missing = df_check.copy()
    # Randomly set some values to NaN
    np.random.seed(42)
    missing_indices = np.random.choice(df_with_missing.index, size=15, replace=False)
    df_with_missing.loc[missing_indices, 'age'] = np.nan
    missing_indices = np.random.choice(df_with_missing.index, size=10, replace=False)
    df_with_missing.loc[missing_indices, 'salary'] = np.nan
    
    missing_count = df_with_missing.isnull().sum().sum()
    missing_per_col = df_with_missing.isnull().sum()
    print(f"\n   📋 Simulating real-world scenario with missing values:")
    print(f"   - Total missing values: {missing_count}")
    print(f"   - Missing values per column:")
    for col, count in missing_per_col.items():
        if count > 0:
            print(f"     • {col}: {count} missing ({count/len(df_with_missing)*100:.1f}%)")

# Check for duplicates
duplicate_count = df_check.duplicated().sum()
print(f"\n⚠️  Duplicates Found:")
if duplicate_count > 0:
    print(f"   - Total duplicate rows: {duplicate_count}")
else:
    print(f"   - No duplicates in current dataset")
    print(f"   - But real-world data often has duplicate records!")

print(f"\n💡 The Problem:")
print(f"   - We've loaded the data successfully")
print(f"   - But the data has quality issues (missing values, potential duplicates)")
print(f"   - We can't proceed with analysis until we clean the data!")
print(f"   - Statistical operations will fail or give wrong results with missing values")
print(f"   - Duplicates can skew our analysis and lead to incorrect conclusions")

print(f"\n➡️  Solution Needed:")
print(f"   - We need techniques to handle missing values")
print(f"   - We need methods to detect and remove duplicates")
print(f"   - This leads us to Example 2: Missing Values & Duplicates")

print("\n" + "=" * 70)


🚫 DEMONSTRATING THE DEAD END: Data Quality Issues

📊 Checking data quality in loaded dataset...
   Dataset shape: 100 rows × 6 columns

⚠️  Missing Values Found:
   - No missing values in current dataset
   - But real-world data often has missing values!

   📋 Simulating real-world scenario with missing values:
   - Total missing values: 25
   - Missing values per column:
     • age: 15 missing (15.0%)
     • salary: 10 missing (10.0%)

⚠️  Duplicates Found:
   - No duplicates in current dataset
   - But real-world data often has duplicate records!

💡 The Problem:
   - We've loaded the data successfully
   - But the data has quality issues (missing values, potential duplicates)
   - We can't proceed with analysis until we clean the data!
   - Statistical operations will fail or give wrong results with missing values
   - Duplicates can skew our analysis and lead to incorrect conclusions

➡️  Solution Needed:
   - We need techniques to handle missing values
   - We need methods to dete

### What We Need Next

**The Solution**: We need data cleaning techniques:
- **Missing value handling**: Fill, drop, or impute missing values
- **Duplicate detection**: Find and remove duplicate rows
- **Data quality checks**: Validate data before analysis

**This dead end leads us to Example 2: Missing Values & Duplicates**
- Example 2 will teach us how to handle missing values
- We'll learn strategies for dealing with duplicates
- This solves the data quality problem so we can proceed with analysis!


## 📚 References

1. McKinney, W. (2010). *Data Structures for Statistical Computing in Python*. Proceedings of the 9th Python in Science Conference (SciPy). <https://doi.org/10.25080/Majora-92bf1922-00a>
2. Wickham, H. (2014). *Tidy Data*. Journal of Statistical Software, 59(10), 1-23. <https://doi.org/10.18637/jss.v059.i10>
3. McKinney, W. (2017). *Python for Data Analysis*, 2nd ed. O'Reilly Media.